# Tiled attention
Can attention avoid allocating the full score matrix?

In [ ]:
import os
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
smoke = os.environ.get('PYTHONIC_SMOKE') == '1'

In [ ]:
from pythonic.attention import causal_attention
from pythonic.tiled import tiled_attention
q, k, v = [torch.randn(1, 4, 47, 16) for _ in range(3)]
reference = causal_attention(q, k, v)
for tile in (8, 16, 32):
    result = tiled_attention(q, k, v, tile)
    print(tile, 'tile:', (result - reference).abs().max().item(), 'maximum error')
    torch.testing.assert_close(result, reference)

In [ ]:
from pythonic.experiments import attention_benchmark
report = attention_benchmark(length=32 if smoke else 256, repeats=3 if smoke else 20)
for name, timing in report['timings'].items():
    print(name, round(timing['median_ms'], 3), 'ms')
print(report['metadata'])

The Python tiled version demonstrates online softmax. SDPA selects an available backend; timings describe this device only.

Reference: [FlashAttention](https://arxiv.org/abs/2205.14135).